In [ ]:
import dotenv
import os 
from pathlib import Path

project_dir = Path('..').resolve()
dotenv_path = project_dir / '.env'
dotenv.load_dotenv(dotenv_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm

videos = list(Path(os.environ['MFE_RAW_VIDEOS']).glob('*.mp4'))
videos = list(filter(lambda x: not x.parts[-1].startswith('.'), videos)) # filter out videos that start with .
print(len(videos))

In [ ]:
output_folder = project_dir / 'reports/figures/lazy_card_detection'
if not output_folder.exists():
    output_folder.mkdir(parents=True)

frames = 5

for video in videos:
    cap = cv2.VideoCapture(str(video))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    for i in range(frames):
        frame_idx = int(i*fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        h, w, c = frame.shape

        separation = 500
        bottom_half = round(frame[separation:].mean(), 3)
        top_half = round(frame[:separation].mean(), 3)
        index_bottom_over_top = round(bottom_half / top_half, 3)

        gs = plt.GridSpec(1, 2)
        plt.figure(figsize=(12, 4))
        plt.subplot(gs[0])
        plt.imshow(frame)
        plt.xlabel('pixels')
        plt.ylabel('pixels')

        ax = plt.subplot(gs[1])
        plt.imshow(frame)
        plt.xlabel('pixels')
        plt.ylabel('pixels')

        plt.text(.01, .99, f'Mean pixel={top_half}', ha='left', va='top', transform=ax.transAxes, fontdict={'color': 'white'})
        plt.text(.01, 1-separation/h, f'Mean pixel={bottom_half}', ha='left', va='top', transform=ax.transAxes, fontdict={'color': 'white'})
        plt.text(.99, .99, f'Index bottom/top={index_bottom_over_top}', ha='right', va='top', transform=ax.transAxes, fontdict={'color': 'white', 'weight': '1000'})
    
        top_half = plt.Polygon([
            [0, 0],
            [w, 0],
            [w, separation],
            [0, separation]
        ], alpha=0.4)
        ax.add_patch(top_half)

        bottom_half = plt.Polygon([
            [0, separation],
            [w, separation],
            [w, h],
            [0, h]
        ], alpha=0.4, color='green')
        ax.add_patch(bottom_half)
        ax.yaxis.set_visible(False)

        fname, ext = video.parts[-1].split('.')

        plt.suptitle(f"{fname}, Frame {frame_idx}")
        plt.tight_layout()
        plt.savefig(output_folder / f"{fname}_frame{frame_idx:05}.png")
        plt.show()    
        
    break

In [ ]:
output_folder = project_dir / 'reports/interim/lazy_cards'
if not output_folder.exists():
    output_folder.mkdir(parents=True)

frames = 60

for video in tqdm(videos):
    try: 
        fname, ext = video.parts[-1].split('.')
        cap = cv2.VideoCapture(str(video))
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        
        best_frame = None
        best_index_bottom_over_top = None
        
        for i in np.arange(frames):
            frame_idx = int(i*fps)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            separation = 500
            bottom_half = round(frame[separation:].mean(), 3)
            top_half = round(frame[:separation].mean(), 3)
            index_bottom_over_top = round(bottom_half / top_half, 3)
            # if bottom_half > 150 and bottom_half < 170 and index_bottom_over_top > 1:
            #     plt.imshow(frame)
            #     plt.show()

            if best_frame is None or index_bottom_over_top > best_index_bottom_over_top and bottom_half < 170:
                best_frame = frame_idx 
                best_index_bottom_over_top = index_bottom_over_top

        cap.set(cv2.CAP_PROP_POS_FRAMES, best_frame)
        ret, frame = cap.read()
        plt.imsave(output_folder / f"{fname}.png", frame)
        # plt.imshow(frame)
        # plt.show()
    except:
        print("error on video", video)
